# Lab: K-Nearest Neighbors (KNN)

## 1. Ý tưởng

KNN là một trong những thuật toán *đơn giản* nhất nhưng cực kỳ trực quan: **"Nói cho tôi biết bạn của bạn là ai, tôi sẽ nói bạn là người thế nào."**

Để dự đoán nhãn của một điểm mới, KNN làm 3 bước:
1. Tính khoảng cách từ điểm mới đến **mọi** điểm trong tập train.
2. Lấy $k$ điểm gần nhất.
3. Bầu chọn nhãn xuất hiện nhiều nhất trong $k$ điểm đó (mode).

$$
\hat{y} = \text{mode}\{y_{(1)}, y_{(2)}, \dots, y_{(k)}\}
$$

Trong đó $y_{(i)}$ là nhãn của hàng xóm gần thứ $i$.

## 2. Đặc điểm độc đáo

- **Không có giai đoạn train**: KNN chỉ "ghi nhớ" tập train. Tất cả tính toán xảy ra lúc dự đoán → train nhanh, **predict chậm** (phải tính khoảng cách với toàn bộ train mỗi lần).
- **Non-parametric**: không giả định phân phối dữ liệu, không có tham số học được.
- **Lazy learner**: trì hoãn mọi tính toán đến lúc cần.

## 3. Chọn k và chọn metric khoảng cách

### Hàm khoảng cách

Phổ biến nhất là **Euclidean** (L2):
$$d(u, v) = \sqrt{\sum_{i=1}^{n}(u_i - v_i)^2}$$

**Manhattan** (L1):
$$d(u, v) = \sum_{i=1}^{n}|u_i - v_i|$$

**Cosine distance** (cho dữ liệu văn bản, vector cao chiều):
$$d_{\cos}(u, v) = 1 - \frac{u \cdot v}{\|u\| \, \|v\|}$$

Lưu ý: **cosine *similarity*** là $\frac{u\cdot v}{\|u\|\|v\|}$ (càng lớn càng gần). **Cosine *distance*** là $1 - \text{similarity}$ (càng nhỏ càng gần). Đây là chỗ rất dễ nhầm — sắp xếp tăng dần cho distance, giảm dần cho similarity.

### Chọn k

- $k = 1$: nhạy nhiễu (nhìn 1 hàng xóm thì đoán theo hàng xóm đó, kể cả khi hàng xóm là outlier).
- $k$ lớn: dự đoán mượt hơn nhưng bias cao (có thể bỏ qua chi tiết).
- Chọn $k$ lẻ cho phân loại 2 lớp để tránh hoà.
- Thực tế: dùng cross-validation để tìm $k$ tối ưu, thường $k \in [3, 30]$.

## 4. Vì sao phải chuẩn hoá feature

KNN dựa trên khoảng cách → feature có scale lớn sẽ *thống trị* khoảng cách. Ví dụ feature `Age` (0-100) và `Income` (0-1.000.000) — chênh nhau 1 đơn vị Income lấn át chênh 1 đơn vị Age.

**Quy tắc**: luôn chuẩn hoá (`StandardScaler` hoặc `MinMaxScaler`) trước khi dùng KNN.

**Nguyên tắc tối quan trọng**: chỉ `fit` scaler trên tập **train**, rồi `transform` lên test. Nếu fit trên cả tập (gồm cả test) → **data leakage** → kết quả test bị thổi phồng giả tạo.

# THỰC HÀNH 1: KNN trên Iris (4 features liên tục)

Dataset kinh điển: 150 hoa Iris, 4 đặc trưng (sepal length/width, petal length/width), 3 loài (setosa, versicolor, virginica). Mục tiêu: phân loại loài.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

np.random.seed(42)

iris = pd.read_excel('data/Iris.xls')
print(f'Shape: {iris.shape}')
print(iris.head())
print(f'\nClasses: {iris.iloc[:, -1].unique()}')

In [ ]:
X = iris.iloc[:, :-1].values
y = LabelEncoder().fit_transform(iris.iloc[:, -1])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# Quan trọng: fit scaler CHỈ trên train, transform trên test.
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')

In [ ]:
# Train KNN với vài giá trị k khác nhau để chọn k tối ưu
ks = list(range(1, 31))
cv_scores = []
for k in ks:
    knn = KNeighborsClassifier(n_neighbors=k)
    scores = cross_val_score(knn, X_train_s, y_train, cv=5, scoring='accuracy')
    cv_scores.append(scores.mean())

best_k = ks[int(np.argmax(cv_scores))]
print(f'Best k theo 5-fold CV: {best_k}, accuracy = {max(cv_scores)*100:.2f}%')

plt.figure(figsize=(8, 4))
plt.plot(ks, [s*100 for s in cv_scores], 'o-')
plt.xlabel('k'); plt.ylabel('CV Accuracy (%)')
plt.title('Chọn k bằng cross-validation')
plt.axvline(best_k, color='red', linestyle='--', label=f'best k={best_k}')
plt.legend(); plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Đánh giá trên tập test với best_k
knn_best = KNeighborsClassifier(n_neighbors=best_k)
knn_best.fit(X_train_s, y_train)
y_pred = knn_best.predict(X_test_s)

print(f'Test accuracy: {accuracy_score(y_test, y_pred)*100:.2f}%')
print()
print(classification_report(y_test, y_pred,
                            target_names=['setosa', 'versicolor', 'virginica']))

## 5. Tự cài đặt KNN từ đầu

Để hiểu KNN đang làm gì, ta tự code phiên bản từ scratch và so sánh với sklearn.

In [ ]:
from collections import Counter

class MyKNN:
    def __init__(self, k=5, metric='euclidean'):
        self.k = k
        self.metric = metric

    def fit(self, X, y):
        # KNN không có giai đoạn train thực sự — chỉ lưu lại dữ liệu.
        self.X_train = np.asarray(X)
        self.y_train = np.asarray(y)
        return self

    def _distance(self, x):
        if self.metric == 'euclidean':
            return np.sqrt(((self.X_train - x) ** 2).sum(axis=1))
        elif self.metric == 'manhattan':
            return np.abs(self.X_train - x).sum(axis=1)
        elif self.metric == 'cosine':
            num = self.X_train @ x
            den = np.linalg.norm(self.X_train, axis=1) * np.linalg.norm(x) + 1e-10
            similarity = num / den
            return 1 - similarity            # cosine DISTANCE (càng nhỏ càng gần)
        else:
            raise ValueError(f'Unknown metric: {self.metric}')

    def predict_one(self, x):
        d = self._distance(x)
        nn_idx = np.argsort(d)[:self.k]      # k khoảng cách NHỎ NHẤT
        labels = self.y_train[nn_idx]
        return Counter(labels).most_common(1)[0][0]

    def predict(self, X):
        return np.array([self.predict_one(np.asarray(x)) for x in X])

    def score(self, X, y):
        return (self.predict(X) == np.asarray(y)).mean()

# So sánh với sklearn
my_knn = MyKNN(k=best_k).fit(X_train_s, y_train)
acc_mine = my_knn.score(X_test_s, y_test)
acc_skl = knn_best.score(X_test_s, y_test)
print(f'My KNN  test acc: {acc_mine*100:.2f}%')
print(f'sklearn test acc: {acc_skl*100:.2f}%')
print('Hai con số phải khớp nhau (lệch nhỏ là do tie-breaking).')

# THỰC HÀNH 2: KNN trên drug200

Dữ liệu trộn cả feature liên tục (Age, Na_to_K) và rời rạc (Sex, BP, Cholesterol). Sau khi encode, ta có không gian số → KNN dùng được.

In [ ]:
drug = pd.read_csv('data/drug200.csv')

# One-hot cho cột rời rạc — giữ tính chất "không có thứ tự" cho Sex, Cholesterol
drug_enc = pd.get_dummies(drug, columns=['Sex', 'BP', 'Cholesterol'], drop_first=False)
X = drug_enc.drop('Drug', axis=1).values.astype(float)
y = LabelEncoder().fit_transform(drug_enc['Drug'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

# Scaling SAU khi đã split — fit chỉ trên train
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

for k in [1, 3, 5, 7, 11, 15]:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_s, y_train)
    acc = knn.score(X_test_s, y_test)
    print(f'k = {k:2d}  test acc = {acc*100:.2f}%')

## 6. Phụ lục: Nearest Centroid (Rocchio)

Một họ hàng đơn giản hơn của KNN: thay vì so với **mọi** điểm train, ta tính **trung tâm (centroid)** của mỗi lớp rồi gán nhãn theo centroid gần nhất.

$$\mu_c = \frac{1}{|S_c|}\sum_{x \in S_c} x, \quad \hat{y}(x) = \arg\min_c \|x - \mu_c\|$$

Ưu điểm: predict O(C) thay vì O(N) như KNN. Nhược: chỉ hoạt động tốt khi mỗi lớp có dạng tròn/lồi và phương sai gần nhau.

In [ ]:
from sklearn.neighbors import NearestCentroid

# Trên Iris
X_train_s_iris = StandardScaler().fit(iris.iloc[:, :-1].values)
y_iris = LabelEncoder().fit_transform(iris.iloc[:, -1])

X_iris = iris.iloc[:, :-1].values
X_tr, X_te, y_tr, y_te = train_test_split(X_iris, y_iris, test_size=0.2,
                                          random_state=42, stratify=y_iris)
scaler_iris = StandardScaler()
X_tr_s = scaler_iris.fit_transform(X_tr)
X_te_s = scaler_iris.transform(X_te)

nc = NearestCentroid()
nc.fit(X_tr_s, y_tr)
print(f'Nearest Centroid trên Iris: {nc.score(X_te_s, y_te)*100:.2f}%')
print(f'KNN(k={best_k}) trên Iris : {knn_best.score(X_test_s, y_test)*100:.2f}%')
print('\nNearest Centroid thường nhanh hơn nhưng kém chính xác hơn KNN khi lớp có hình dạng phức tạp.')

## Tổng kết

1. KNN dự đoán bằng cách bầu chọn từ $k$ hàng xóm gần nhất.
2. **Phải scale feature** trước khi dùng (vì KNN dùng khoảng cách).
3. **Phải fit scaler chỉ trên train**, rồi transform test — tránh data leakage.
4. Chọn $k$ bằng cross-validation, không phải tuỳ ý.
5. Phân biệt **cosine similarity** (lớn = gần) và **cosine distance = 1 − similarity** (nhỏ = gần).
6. Nearest Centroid là họ hàng đơn giản, nhanh hơn nhưng kém linh hoạt.

# BÀI TẬP VỀ NHÀ

## Bài 1: So sánh metric khoảng cách
Trên Iris, train `KNeighborsClassifier(n_neighbors=5)` với 3 metric: `euclidean`, `manhattan`, `chebyshev`. So sánh test accuracy.

*Gợi ý:* `KNeighborsClassifier(n_neighbors=5, metric='manhattan')`.

## Bài 2: Trọng số khoảng cách
Mặc định KNN đếm hàng xóm bằng phiếu bằng nhau. Có thể đổi sang trọng số `weights='distance'` — hàng xóm gần hơn có quyền hơn. Trên drug200, so sánh `weights='uniform'` vs `weights='distance'` với k = 5, 11, 15. Cái nào tốt hơn? Vì sao?

## Bài 3: Vẽ decision boundary
Lấy 2 feature đầu của Iris (sepal length, sepal width). Train KNN với k = 1, 5, 30. Vẽ decision boundary cho mỗi k bằng `plt.contourf` trên lưới điểm. Quan sát: k nhỏ → boundary lởm chởm (overfit), k lớn → mượt nhưng có thể bỏ chi tiết.

*Gợi ý:* tạo lưới `np.meshgrid`, predict lưới, vẽ contour.

## Bài 4: KNN cho text
Trên dataset `Education.csv` (đã dùng ở lab Naive Bayes), thử dùng KNN. Yêu cầu:
1. `CountVectorizer` để vector hoá văn bản.
2. KNN với `metric='cosine'`.
3. Sweep $k \in \{1, 3, 5, 7\}$.
4. So sánh với Multinomial NB (đã có ở bài Naive Bayes). Kết luận?

## Bài 5: Curse of dimensionality
Sinh dữ liệu giả: 100 mẫu, $d$ chiều, ngẫu nhiên Gaussian, 2 lớp. Cho $d \in \{2, 10, 50, 200\}$, đo accuracy của KNN(k=5). Quan sát: khi $d$ tăng, KNN kém dần. Đây là **lời nguyền chiều cao**: ở chiều cao, mọi điểm xa nhau gần như nhau → khái niệm "hàng xóm gần" không còn ý nghĩa.